# Session 3 (Thu Jul 30) — Tool calling + a well-composed chatbot

Your bot already answers questions from the NRP docs. Today we give it **hands**: it can
call tools to fetch live cluster state, then combine that with retrieved docs for a
grounded answer. We also polish the UX: streaming answers, a sources expander, and
graceful handling of out-of-scope questions.

By the end you'll have a bot that:
1. Decides **when** to call a tool vs. answer from docs alone
2. Calls **at most once per tool** and then writes the final answer
3. Shows sources inline and handles "I don't know" cleanly

> This is the live-demo polish. The poster is already frozen — these improvements won't
> change your figures, but they'll make the showcase impressive.

## Setup

Same token and models as Sessions 1–2. We'll use `gpt-oss` for chat (fast, reliable) and
`qwen3-embedding` for retrieval.

In [ ]:
import os, json, logging
from dotenv import load_dotenv
from openai import OpenAI
import chromadb

load_dotenv()
logging.getLogger("chromadb.telemetry.product.posthog").setLevel(logging.CRITICAL)

client = OpenAI(
    api_key=os.environ["NRP_LLM_TOKEN"],
    base_url=os.environ.get("NRP_LLM_BASE_URL", "https://ellm.nrp-nautilus.io/v1"),
    timeout=120,
)
CHAT_MODEL = "gpt-oss"
EMBED_MODEL = "qwen3-embedding"

coll = chromadb.PersistentClient(path="./chroma_db").get_or_create_collection("nrp_docs")
print(f"index ready: {coll.count()} chunks")

## A. Tools — giving the bot hands

Two tools, both read-only:

| Tool | What it does | When to use it |
|---|---|---|
| `search_nrp_docs(query)` | Returns top-5 doc chunks for a query | Any how-to / concept question about NRP |
| `run_kubectl(verb, resource)` | Runs `kubectl verb resource -n mfsada` | Live cluster state: pods, services, etc. |

The critical rule: **call each tool at most once**, then answer from the results. No
re-searching to refine — that loops forever.

In [ ]:
def embed(text):
    return client.embeddings.create(model=EMBED_MODEL, input=[text]).data[0].embedding

def search_nrp_docs(query, k=5):
    r = coll.query(query_embeddings=[embed(query)], n_results=k)
    return "\n\n".join(
        f"[Source: {m['title']} | {m['source_url']}]\n{d[:800]}"
        for d, m in zip(r["documents"][0], r["metadatas"][0])
    )

ALLOWED_VERBS = {"get", "describe", "top"}
ALLOWED_RES  = {"pods", "svc", "services", "deployments", "deploy", "pvc", "nodes", "ingress"}

def run_kubectl(verb, resource):
    verb = verb.strip().split()[0]
    resource = resource.strip().split()[0]
    if verb not in ALLOWED_VERBS:
        return f"refused: verb '{verb}' not allowed (try get/describe/top)"
    if resource not in ALLOWED_RES:
        return f"refused: resource '{resource}' not allowed"
    import subprocess
    out = subprocess.run(
        ["kubectl", verb, resource, "-n", "mfsada"],
        capture_output=True, text=True, timeout=30
    )
    return (out.stdout or out.stderr)[:1500]

# Quick smoke test
print("kubectl test:", run_kubectl("get", "pods")[:120])
print("docs test:", search_nrp_docs("GPU request")[:120])

### A1. The tool loop — call once, then answer

The model gets a system prompt that enforces the rule. We run at most 3 rounds:
- Round 1–2: tools allowed
- Round 3: **no tools** — forces a final answer even if the model wants to search again

In [ ]:
TOOLS = [
    {"type": "function", "function": {
        "name": "search_nrp_docs",
        "description": "Search NRP documentation. Use for how-to and concept questions.",
        "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}
    }},
    {"type": "function", "function": {
        "name": "run_kubectl",
        "description": "Read-only kubectl against the user's namespace. Use for live cluster state.",
        "parameters": {"type": "object", "properties": {"verb": {"type": "string"}, "resource": {"type": "string"}}, "required": ["verb", "resource"]}
    }}
]

SYSTEM = ("You are the NRP assistant. Use search_nrp_docs for documentation questions and "
          "run_kubectl for live cluster state. Call each tool at most once. As soon as you "
          "have tool results, write the final answer — do not search again to refine. "
          "Cite sources when you use the docs. If the tools don't cover it, say so honestly.")

REGISTRY = {"search_nrp_docs": search_nrp_docs, "run_kubectl": run_kubectl}

def chat_with_tools_user(user_text, max_rounds=3):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user_text}]
    used = set()
    for rnd in range(max_rounds):
        kw = {"tools": TOOLS} if rnd < max_rounds - 1 else {}
        r = client.chat.completions.create(model=CHAT_MODEL, messages=msgs, **kw)
        m = r.choices[0].message
        if not m.tool_calls:
            return m.content or "(no answer returned)"
        msgs.append({"role": "assistant", "content": m.content or "", "tool_calls": [
            {"id": tc.id, "type": "function", "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
            for tc in m.tool_calls
        ]})
        for tc in m.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments)
            if name in used:
                out = "Already called. Answer from the results you have."
            else:
                used.add(name)
                try:
                    out = REGISTRY[name](**args)
                except Exception as e:
                    out = f"error: {e}"
            print(f"   -> {name}({args}) [{len(str(out))} chars]")
            msgs.append({"role": "tool", "tool_call_id": tc.id, "content": str(out)})
    return "(hit max rounds without final answer)"

# Test it
print("\n=== Q: What pods are running? ===")
print(chat_with_tools_user("What pods are running in my namespace?")[:500])

### A2. RAG + tools together

Some questions need both: fetch live state, then ground the explanation in docs. The same
loop handles it — the model decides which tool(s) to call.

In [ ]:
print("\n=== Q: How do I request a GPU? ===")
print(chat_with_tools_user("How do I request a GPU on Nautilus?")[:600])

## B. Polishing the chatbot

The loop above works in a notebook. For a showcase demo we want:
- **Streaming** answers (no long pauses)
- **Sources expander** (click to see retrieved docs)
- **Honesty exit** (graceful "I don't know")

All of this fits in ~60 lines of Streamlit.

In [ ]:
%%writefile app_with_tools.py
import os, json, logging
logging.getLogger("chromadb.telemetry.product.posthog").setLevel(logging.CRITICAL)
import streamlit as st, chromadb
from openai import OpenAI

DB = os.environ.get("CHROMA_PATH", "./chroma_db")  # pod sets CHROMA_PATH=/data/chroma_db
client = OpenAI(
    api_key=os.environ["NRP_LLM_TOKEN"],
    base_url=os.environ.get("NRP_LLM_BASE_URL", "https://ellm.nrp-nautilus.io/v1"),
    timeout=120,
)
CHAT_MODEL = os.environ.get("CHAT_MODEL", "gpt-oss")
EMBED_MODEL = os.environ.get("EMBED_MODEL", "qwen3-embedding")

@st.cache_resource
def get_collection():
    return chromadb.PersistentClient(path=DB).get_or_create_collection("nrp_docs")

coll = get_collection()

def embed(t):
    return client.embeddings.create(model=EMBED_MODEL, input=[t]).data[0].embedding

def search_nrp_docs(query, k=5):
    r = coll.query(query_embeddings=[embed(query)], n_results=k)
    return [{"text": d, "source_url": m["source_url"], "title": m["title"]}
            for d, m in zip(r["documents"][0], r["metadatas"][0])]

ALLOWED_VERBS = {"get", "describe", "top"}
ALLOWED_RES  = {"pods", "svc", "services", "deployments", "deploy", "pvc", "nodes", "ingress"}

def run_kubectl(verb, resource):
    verb = verb.strip().split()[0]
    resource = resource.strip().split()[0]
    if verb not in ALLOWED_VERBS or resource not in ALLOWED_RES:
        return "refused: that call is not allowed"
    import subprocess
    out = subprocess.run(
        ["kubectl", verb, resource, "-n", "mfsada"],
        capture_output=True, text=True, timeout=30
    )
    return (out.stdout or out.stderr)[:1500]

TOOLS = [
    {"type": "function", "function": {
        "name": "search_nrp_docs",
        "description": "Search NRP documentation. Use for how-to and concept questions.",
        "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}
    }},
    {"type": "function", "function": {
        "name": "run_kubectl",
        "description": "Read-only kubectl against the user's namespace. Use for live cluster state.",
        "parameters": {"type": "object", "properties": {"verb": {"type": "string"}, "resource": {"type": "string"}}, "required": ["verb", "resource"]}
    }}
]

SYSTEM = ("You are the NRP assistant. Use search_nrp_docs for documentation questions and "
          "run_kubectl for live cluster state. Call each tool at most once. As soon as you "
          "have tool results, write the final answer — do not search again to refine. "
          "Cite sources when you use the docs. If the tools don't cover it, say so honestly.")

REGISTRY = {"search_nrp_docs": lambda q: search_nrp_docs(q), "run_kubectl": run_kubectl}

def chat_with_tools(user_text, max_rounds=3):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user_text}]
    used = set()
    tool_results = []
    for rnd in range(max_rounds):
        kw = {"tools": TOOLS} if rnd < max_rounds - 1 else {}
        r = client.chat.completions.create(model=CHAT_MODEL, messages=msgs, **kw)
        m = r.choices[0].message
        if not m.tool_calls:
            return m.content or "(no answer returned)", tool_results
        msgs.append({"role": "assistant", "content": m.content or "", "tool_calls": [
            {"id": tc.id, "type": "function", "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
            for tc in m.tool_calls
        ]})
        for tc in m.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments)
            if name in used:
                out = "Already called. Answer from the results you have."
            else:
                used.add(name)
                try:
                    out = REGISTRY[name](**args)
                except Exception as e:
                    out = f"error: {e}"
            tool_results.append({"name": name, "args": args, "result": out})
            msgs.append({"role": "tool", "tool_call_id": tc.id, "content": str(out)})
    return "(hit max rounds without final answer)", tool_results

st.set_page_config(page_title="NRP Bot + Tools", page_icon="🤖")
st.title("🤖 NRP Bot + Tools")
st.caption(f"Index: {coll.count()} chunks | Tools: search_nrp_docs, run_kubectl")

if "messages" not in st.session_state:
    st.session_state.messages = []
for m in st.session_state.messages:
    if m["role"] != "system":
        st.chat_message(m["role"]).write(m["content"])

if prompt := st.chat_input("Ask about NRP..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    st.chat_message("user").write(prompt)
    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            answer, tools = chat_with_tools(prompt)
        st.write(answer)
        if tools:
            with st.expander("Tool calls & sources"):
                for t in tools:
                    st.json({"tool": t["name"], "args": t["args"], "result_preview": str(t["result"])[:300]})
    st.session_state.messages.append({"role": "assistant", "content": answer})

### B1. Run it locally

```bash
streamlit run app_with_tools.py
```

Try:
- "What pods are running in my namespace?" → calls kubectl
- "How do I request a GPU?" → searches docs
- "What is the airspeed velocity of an unladen swallow?" → honest "I don't know"

## C. Showcase readiness

Your poster is frozen. This session's improvements are for the **live demo**:
- The bot calls tools visibly (judges love seeing `kubectl` run live)
- It cites sources inline
- It admits ignorance cleanly

Practice your 90-second pitch with these live demos ready. The system is real, running on
NRP, and you built it end to end.